# Validate data in silver zone

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from dotenv import load_dotenv
import os

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # jar files supporting read/write postgres, trino, S3, hive (delta table for lake house)
    .config(
        'spark.jars',
        """
            ../../../../spark/jar/trino-jdbc-480.jar,
            ../../../../spark/jar/hadoop-aws-3.4.2.jar,
            ../../../../spark/jar/bundle-2.29.52.jar,
            ../../../../spark/jar/delta-spark_4.1_2.13-4.2.0.jar,
            ../../../../spark/jar/delta-kernel-api-4.2.0.jar,
            ../../../../spark/jar/delta-kernel-defaults-4.2.0.jar,
            ../../../../spark/jar/delta-storage-4.2.0.jar,
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark

26/06/24 09:42:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
from pyspark.sql import SparkSession

def post_valid_silver(spark:SparkSession, start_date:str, end_date:str):
    trino_url = "jdbc:trino://localhost:8082"
    trino_user = "trino"
    trino_password = ""
    source_table = "datalake.bronze.nyc_taxi"

    df_bronze = (
        spark.read
        .option("pushDownPredicate", "false")
        .format("jdbc")
        .option("url", trino_url)
        .option("driver", "io.trino.jdbc.TrinoDriver")
        .option("dbtable", source_table)
        .option("user", trino_user)
        .option("password", trino_password)
        .load()
    )

    silver_table = 'silver.nyc_taxi'
    df_silver = spark.read.table(silver_table)

    count_bronze = df_bronze.filter(
        (df_bronze['pickup_datetime'] <= end_date) &
        (df_bronze['pickup_datetime'] > start_date)
    ).dropDuplicates().count()

    count_silver = df_silver.filter(
        (df_silver['pickup_datetime'] <= end_date) &
        (df_silver['pickup_datetime'] > start_date)
    ).count()

    print(f'Count bronze: {count_bronze}\nCount silver: {count_silver}')
    assert count_bronze == count_silver
    print('\n=== Test pass ===\n')

- Valid by function

In [4]:
post_valid_silver(spark, '2010-05-31 23:57:57', '2010-09-01 00:23:00')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/06/24 07:18:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Count bronze: 419976
Count silver: 419975


AssertionError: 

- Check from 2010-05-31 23:57:57 to 2010-09-01 00:23:00

In [ ]:
# load silver
df_silver = spark.read.table("silver.nyc_taxi")
df_silver.filter(
    (df_silver.pickup_datetime > '2010-05-31 23:57:57') &
    (df_silver.pickup_datetime <= '2010-09-01 00:23:00')
).count()

419975

In [10]:
df_silver.select(F.max(F.col('pickup_datetime'))).show()

+--------------------+
|max(pickup_datetime)|
+--------------------+
| 2010-09-01 00:23:00|
+--------------------+



In [7]:
# load bronze
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""
source_table = "datalake.bronze.nyc_taxi"

df_bronze = (
    spark.read
    .option("pushDownPredicate", "false")
    .format("jdbc")
    .option("url", trino_url)
    .option("driver", "io.trino.jdbc.TrinoDriver")
    .option("dbtable", source_table)
    .option("user", trino_user)
    .option("password", trino_password)
    .load()
)

df_bronze = df_bronze.filter(
    (df_bronze.pickup_datetime > '2010-05-31 23:57:57') &
    (df_bronze.pickup_datetime <= '2010-09-01 00:23:00')
)
df_bronze.count()

420001

In [11]:
df_bronze.select(F.max(F.col('pickup_datetime'))).show()

+--------------------+
|max(pickup_datetime)|
+--------------------+
| 2010-09-01 00:23:00|
+--------------------+



- Transform bronze data and check

In [9]:
import sys
sys.path.append('../../../')

from silver.silver_map import payment_map, store_fwd_flag_map, vendor_id_map

# transfer: cast some columns, replace data within some columns
cast_map = {
    'passenger_count': F.col('passenger_count').cast(T.IntegerType()),
    'pu_location_id' : F.col('pu_location_id').cast(T.IntegerType()),
    'do_location_id' : F.col('do_location_id').cast(T.IntegerType())
}
df = df_bronze.dropDuplicates().withColumns(cast_map)
df = (
    df
    .withColumn('payment_type', F.lower(F.trim(F.col('payment_type'))))
    .replace(payment_map, subset='payment_type')
)
df = (
    df
    .withColumn('store_and_fwd_flag', F.lower(F.trim(F.col('store_and_fwd_flag'))))
    .replace(store_fwd_flag_map, subset='store_and_fwd_flag')
)
df = (
    df
    .withColumn('vendor_id', F.lower(F.trim(F.col('vendor_id'))))
    .replace(vendor_id_map, subset='vendor_id')
)

# de-dup
df = (
    df
    .dropDuplicates()
)

df.count()

419976

In [10]:
df_bronze.filter(
    (df_bronze['pickup_datetime'] <= '2010-09-01 00:23:00') &
    (df_bronze['pickup_datetime'] > '2010-05-31 23:57:57')
).count()

420001

In [15]:
df_bronze = df_bronze.filter(
    (df_bronze['pickup_datetime'] <= '2010-09-01 00:23:00') &
    (df_bronze['pickup_datetime'] > '2010-05-31 23:57:57')
).dropDuplicates()
df_bronze.count()

419976

In [12]:
df_silver = spark.read.table('silver.nyc_taxi')
df_silver = df_silver.filter(
    (df_silver['pickup_datetime'] > '2010-05-31 23:57:57') & (df_silver['pickup_datetime'] <= '2010-09-01 00:23:00')
)
df_silver.count()

419975

In [17]:
df_bronze.select(F.min(F.col('pickup_datetime'))).show()

+--------------------+
|min(pickup_datetime)|
+--------------------+
| 2010-06-01 00:00:32|
+--------------------+



In [18]:
df_silver.select(F.min(F.col('pickup_datetime'))).show()

+--------------------+
|min(pickup_datetime)|
+--------------------+
| 2010-06-01 00:00:32|
+--------------------+



In [19]:
df_silver.dropDuplicates().count()

419975

---

In [8]:
df_good = spark.read.format("delta") \
    .option("versionAsOf", 12) \
    .load('s3a://flezi-data/silver')

In [9]:
df_good.show()

Py4JJavaError: An error occurred while calling o102.showString.
: java.lang.NoSuchMethodError: 'void org.apache.spark.sql.execution.datasources.parquet.ParquetToSparkSchemaConverter.<init>(boolean, boolean, boolean, boolean, boolean, boolean, boolean)'
	at org.apache.spark.sql.delta.Snapshot$.getParquetFileSchemaAndRowCount(Snapshot.scala:892)
	at org.apache.spark.sql.delta.CheckpointProvider$.$anonfun$getParquetSchema$1(CheckpointProvider.scala:195)
	at org.apache.spark.sql.delta.metering.DeltaLogging.recordFrameProfile(DeltaLogging.scala:171)
	at org.apache.spark.sql.delta.metering.DeltaLogging.recordFrameProfile$(DeltaLogging.scala:169)
	at org.apache.spark.sql.delta.CheckpointProvider$.recordFrameProfile(CheckpointProvider.scala:95)
	at org.apache.spark.sql.delta.metering.DeltaLogging.$anonfun$recordDeltaOperationInternal$1(DeltaLogging.scala:139)
	at com.databricks.spark.util.DatabricksLogging.recordOperation(DatabricksLogging.scala:128)
	at com.databricks.spark.util.DatabricksLog

In [4]:
df_silver = spark.read.table('silver.nyc_taxi')
df_silver.createOrReplaceTempView('nyc_taxi')
df_silver = spark.sql('select * from nyc_taxi for version as of 12')
df_silver.select(F.max(F.col('pickup_datetime'))).show()

26/06/24 08:16:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


AnalysisException: [UNSUPPORTED_FEATURE.TIME_TRAVEL] The feature is not supported: Time travel on the relation: `nyc_taxi`. SQLSTATE: 0A000

In [4]:
post_valid_silver(spark, '2009-09-30 23:59:44', '2009-10-31 23:59:00')

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/06/24 09:43:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Count bronze: 139784
Count silver: 139784

=== Test pass ===

